In [194]:
import numpy as np
from pathlib import Path 
import matplotlib.pyplot as plt
import os
import pandas as pd

In [195]:
true_mi = {20: 5,
           40: 10, 
           80: 20, 
           160: 40, 
           320: 80}

## Get metrics for scRatio

In [190]:
result_dir = Path("../../project_folder/results/mi_experiments/scRatio/simulate_under_uncond_sweep/")

results = {"Config": [], 
           "MSE_test": [],
           "MSE_val": [], 
           "Dimensions": [], 
           "Run": []
          }


for file in os.listdir(result_dir):
    for dim in [20, 40, 80, 160, 320]:
        # Get val and test ratios 
        ratios_val = np.load(result_dir / file / f"ratios_{dim}_val.npy")
        ratios_test = np.load(result_dir / file / f"ratios_{dim}_test.npy")
        # Split into three
        ratios_split_val = np.split(ratios_val, 3)
        ratios_split_test = np.split(ratios_test, 3)
        
        split_avgs_val = [arr.mean() for arr in ratios_split_val]
        split_avgs_test = [arr.mean() for arr in ratios_split_test]
        
        # MSE with real values 
        split_avgs_val_mse = [np.abs(avg - true_mi[dim]) for avg in split_avgs_val]
        split_avgs_test_mse = [np.abs(avg - true_mi[dim]) for avg in split_avgs_test]
        for i in range(len(split_avgs_val_mse)):
            results["Config"].append(file)
            results["MSE_val"].append(split_avgs_val_mse[i])
            results["MSE_test"].append(split_avgs_test_mse[i])
            results["Dimensions"].append(dim)
            results["Run"].append(i)

In [191]:
results_df = pd.DataFrame(results)
results_df.groupby(["Config", "Dimensions"]).mean()

MSE_test    MSE_val  Run
Config          Dimensions                           
deterministic   20           0.048347   0.047935  1.0
                40           0.053398   0.043350  1.0
                80           0.051510   0.050333  1.0
                160          0.164042   0.150202  1.0
                320         21.182093  21.090273  1.0
sigmamin_0.01   20           0.043664   0.068197  1.0
                40           0.061230   0.051906  1.0
                80           0.069777   0.060178  1.0
                160          0.133917   0.026206  1.0
                320         20.854738  20.781925  1.0
sigmamin_0.1    20           0.043552   0.057800  1.0
                40           0.030456   0.028866  1.0
                80           0.024389   0.012668  1.0
                160          0.164224   0.082579  1.0
                320         21.000315  20.946859  1.0
stochastic_0.1  20           0.028488   0.033410  1.0
                40           0.051957   0.055336  1.0
                80           0.026578   0.029005  1.0
                160          0.259168   0.234187  1.0
                320         14.579456  14.442523  1.0
stochastic_0.25 20           0.027236   0.040407  1.0
                40           0.093585   0.093981  1.0
                80           0.071206   0.079332  1.0
                160          0.105029   0.146282  1.0
                320          1.157954   1.241575  1.0
stochastic_0.5  20           0.019956   0.027286  1.0
                40           0.217669   0.254589  1.0
                80           0.311665   0.322271  1.0
                160          0.179095   0.138977  1.0
                320          1.096039   0.949196  1.0

In [192]:
results_df.groupby(["Config", "Dimensions"]).std() / np.sqrt(3)

MSE_test   MSE_val      Run
Config          Dimensions                             
deterministic   20          0.018924  0.031030  0.57735
                40          0.020265  0.008825  0.57735
                80          0.025177  0.025236  0.57735
                160         0.045341  0.046130  0.57735
                320         0.137631  0.128294  0.57735
sigmamin_0.01   20          0.013453  0.014492  0.57735
                40          0.033590  0.022014  0.57735
                80          0.034594  0.025879  0.57735
                160         0.019973  0.010259  0.57735
                320         0.045957  0.087593  0.57735
sigmamin_0.1    20          0.020456  0.025572  0.57735
                40          0.018748  0.014548  0.57735
                80          0.009211  0.002311  0.57735
                160         0.066919  0.061354  0.57735
                320         0.260600  0.266705  0.57735
stochastic_0.1  20          0.012318  0.021736  0.57735
                40          0.020143  0.021511  0.57735
                80          0.017515  0.000559  0.57735
                160         0.098208  0.141585  0.57735
                320         2.090012  2.123522  0.57735
stochastic_0.25 20          0.012435  0.013443  0.57735
                40          0.026821  0.045427  0.57735
                80          0.043744  0.049857  0.57735
                160         0.031021  0.062660  0.57735
                320         0.268263  0.253932  0.57735
stochastic_0.5  20          0.007249  0.007990  0.57735
                40          0.092419  0.082423  0.57735
                80          0.024924  0.032523  0.57735
                160         0.055980  0.111642  0.57735
                320         0.348399  0.326402  0.57735

## Results baseline DRECFM

Results for the competing models were obtained using the code at https://github.com/ksnxr/dre-prob-paths using the "TwoSB" option. The adapted repo will be made available upon request, message alessandro.palma@helmholtz-munich.de.

In [160]:
path_dre_cfm = Path("../../project_folder/results/mi_experiments/dre-cfm/dre-cfm_sb_retrains/")

In [161]:
results_baseline_cfm  = {"MSE": [],
                        "Dimensions": [], 
                        "Run": []
                       }

In [162]:
for file in os.listdir(path_dre_cfm):
    file_split = file.split("_")  
    dimensions, rep = int(file_split[4]), int(file_split[6])
    results_baseline_cfm["Run"].append(rep)
    results_baseline_cfm["Dimensions"].append(dimensions)
    
    ratios_cfm = np.load(path_dre_cfm / file / "metrics" / "estim_logr.npy")
    ratio_mean = ratios_cfm.mean()
    results_baseline_cfm["MSE"].append(np.abs(ratio_mean - true_mi[dimensions]))

In [163]:
results_baseline_cfm_df = pd.DataFrame(results_baseline_cfm)

In [164]:
results_baseline_cfm_df.groupby("Dimensions").mean()

,MSE,Run
Dimensions,,
20,0.061192,40.333333
40,0.085309,40.333333
80,0.230340,40.333333
160,0.872691,40.333333
320,2.154556,40.333333


In [165]:
results_baseline_cfm_df.groupby("Dimensions").std() / np.sqrt(3)

,MSE,Run
Dimensions,,
20,0.008241,22.805945
40,0.043984,22.805945
80,0.034371,22.805945
160,0.101788,22.805945
320,0.078399,22.805945


## Results baseline TSM

In [181]:
path_dre_tsm = Path("../../project_folder/results/mi_experiments/dre-cfm/dre-cfm_tsm_retrains/")

In [182]:
results_baseline_tsm = {"MSE": [],
                        "Dimensions": [], 
                        "Run": []
                       }

In [183]:
for file in os.listdir(path_dre_tsm):
    file_split = file.split("_")  
    dimensions, rep = int(file_split[4]), int(file_split[6])
    results_baseline_tsm["Run"].append(rep)
    results_baseline_tsm["Dimensions"].append(dimensions)
    
    ratios_cfm = np.load(path_dre_tsm / file / "metrics" / "estim_logr.npy")
    ratio_mean = ratios_cfm.mean()
    results_baseline_tsm["MSE"].append(np.abs(ratio_mean - true_mi[dimensions]))

In [184]:
results_baseline_tsm_df = pd.DataFrame(results_baseline_tsm)

In [185]:
results_baseline_tsm_df.groupby("Dimensions").mean()

,MSE,Run
Dimensions,,
20,0.034667,40.333333
40,0.382803,40.333333
80,0.250295,40.333333
160,0.888362,40.333333
320,3.546334,40.333333


In [186]:
results_baseline_tsm_df.groupby("Dimensions").std() / np.sqrt(3)

,MSE,Run
Dimensions,,
20,0.010257,22.805945
40,0.023554,22.805945
80,0.101173,22.805945
160,0.108121,22.805945
320,0.120376,22.805945


## Results baseline classifier 

In [221]:
path_clf_results = Path("../../project_folder/results/mi_experiments/classifier/mi")

In [224]:
results_baseline_clf = {"MSE_test": [],
                        "MSE_val": [],
                        "Dimensions": [], 
                        "Hidden dim": [],
                        "Hidden layers": []
                       }

In [235]:
for file in os.listdir(path_clf_results):
    file_split = file.split("_")
    input_dim, hidden_dim, hidden_layers = file_split
    input_dim, hidden_dim, hidden_layers = int(input_dim), int(hidden_dim), int(hidden_layers)
    
    results_path = path_clf_results / file
    log_ratios_val = np.load(results_path / "log_ratios_val.npy")
    log_ratios_test = np.load(results_path / "log_ratios_test.npy")

    log_ratios_val_mean = log_ratios_val.mean()
    log_ratios_test_mean = log_ratios_test.mean()
    mse_val = np.abs(log_ratios_val_mean - true_mi[input_dim])
    mse_test = np.abs(log_ratios_test_mean - true_mi[input_dim])

    print(input_dim)
    print(mse_test.mean())
    
    results_baseline_clf["MSE_test"].append(mse_test)
    results_baseline_clf["MSE_val"].append(mse_val)
    results_baseline_clf["Dimensions"].append(input_dim)
    results_baseline_clf["Hidden dim"].append(hidden_dim)
    results_baseline_clf["Hidden layers"].append(hidden_layers)

20
1.4988794
80
7.6799183
160
0.8361244
160
6.8160973
320
36.52229
320
67.56239
160
5.6112175
20
18.774187
80
6.941843
160
0.5028534
40
14.851866
40
13.840618
20
0.2328167
40
28.444965
40
20.406376
80
24.195908
160
5.2636566
320
52.148605
160
13.68148
160
17.26117
80
18.29295
320
50.218132
160
0.9079704
320
15.625191
20
4.7767735
20
0.8946519
40
12.632996
320
52.08899
80
17.14735
20
22.488771
80
16.004314
20
23.604887
40
17.356052
80
3.1250305
160
0.57407
20
27.281292
80
6.973934
320
43.110325
40
34.328907
320
44.87684
40
14.922472
320
49.861977
40
10.56945
20
0.6333308
80
0.4461403


In [236]:
results_baseline_clf_df = pd.DataFrame(results_baseline_clf)

In [237]:
results_baseline_clf_df.groupby(["Hidden dim", "Hidden layers", "Dimensions"]).mean()

MSE_test    MSE_val
Hidden dim Hidden layers Dimensions                      
64         2             20           0.232817   0.230506
                         40          10.569450  10.555677
                         80          16.004314  15.887054
                         160          0.907970   0.754349
                         320         44.876839  45.034809
           3             20           0.633331   0.612767
                         40          14.851866  14.905956
                         80           3.125031   3.144930
                         160          0.836124   0.971470
                         320         52.088989  51.933392
           4             20           0.894652   0.908343
                         40          13.840618  13.864073
                         80           6.973934   6.942568
                         160         17.261169  17.267031
                         320         66.761932  66.734612
256        2             20           1.498879   1.455011
                         40          28.444965  28.538326
                         80          24.195908  24.239422
                         160          5.611217   5.650711
                         320         49.264095  49.271385
           3             20          18.774187  18.614719
                         40          17.356052  17.297840
                         80           7.679918   7.644022
                         160          5.263657   5.342915
                         320         47.466335  47.347439
           4             20          23.604887  23.380907
                         40          20.406376  20.409481
                         80           0.446140   0.449915
                         160         13.681480  13.645935
                         320         45.999741  46.138756
512        2             20           4.776773   4.728498
                         40          34.328907  34.413090
                         80          17.147350  17.239132
                         160          6.816097   6.750320
                         320         49.301754  49.285995
           3             20          22.488771  22.297476
                         40          12.632996  12.676937
                         80           6.941843   6.884701
                         160          0.502853   0.529892
                         320         35.176865  35.099533
           4             20          27.281292  26.956797
                         40          14.922472  15.015732
                         80          18.292950  18.248348
                         160          0.574070   0.511921
                         320         23.590780  23.631567